In [84]:
import pandas as pd
import numpy as np
# sanity check
non_vpn1 = pd.read_pickle("data/nonvpn1_files_AL2.pkl")
non_vpn2 = pd.read_pickle("data/nonvpn2_files_AL2.pkl")
non_vpn3 = pd.read_pickle("data/nonvpn3_files_AL2.pkl")
vpn      = pd.read_pickle("data/vpn_files_AL2.pkl")
tor      = pd.read_pickle("data/tor_files_AL2.pkl")

df_combined = pd.concat([non_vpn1, non_vpn2, non_vpn3, vpn, tor], ignore_index=True)


In [85]:
def get_flow_duration(packet_seq):
    if len(packet_seq) < 2:
        return 0
    timestamps = [p[0] for p in packet_seq]
    return max(timestamps) - min(timestamps)

df_clean = df_combined.copy()
df_clean["flow_duration"] = df_clean["packet_seq"].apply(get_flow_duration)
df_clean["packet_count"] = df_clean["packet_seq"].apply(len)


In [86]:
# Confiq (from sidd)
# ---------------------------------------------------------------------------
# LOG-SPACED BUCKETING (replaces the uniform bucketing cells)
# Same feature families and column names as before, so the EDA cells still work:
#   bytes_out_i / bytes_in_i, packets_out_i / packets_in_i,
#   gap_mean_i / gap_std_i,  size_mean_i / size_std_i
# ---------------------------------------------------------------------------

# --- CONFIG ---------------------------------------------------------------
BUCKET_STYLE = "log"   # "log" or "uniform"
T_MIN       = 0.01     # end of the first bucket in seconds (bucket 0 = 0 to T_MIN)
T_MAX       = 2    # Z: end of the last bucket in seconds; later packets are dropped
NUM_BUCKETS = 10      # X: number of buckets (more buckets = finer, smaller ratio between edges)

# # edges: [0, T_MIN, ..., T_MAX]  ->  NUM_BUCKETS buckets
# BUCKET_EDGES = np.concatenate([[0.0], np.geomspace(T_MIN, T_MAX, NUM_BUCKETS)])
# assert len(BUCKET_EDGES) - 1 == NUM_BUCKETS

# bucket_info = pd.DataFrame({
#     "bucket": np.arange(NUM_BUCKETS),
#     "start_s": BUCKET_EDGES[:-1],
#     "end_s": BUCKET_EDGES[1:],
# })
# bucket_info["width_s"] = bucket_info["end_s"] - bucket_info["start_s"]
# print(f"ratio between consecutive edges: {BUCKET_EDGES[2] / BUCKET_EDGES[1]:.3f}")
# display(bucket_info.round(3))
if BUCKET_STYLE == "log":
    # edges: 0, 0.01, 0.018, 0.032 ... 1.1, 2.0
    BUCKET_EDGES = np.concatenate([[0.0], np.geomspace(T_MIN, T_MAX, NUM_BUCKETS)])
elif BUCKET_STYLE == "uniform":
    # edges: 0, 0.2, 0.4 ... 2.0
    BUCKET_EDGES = np.linspace(0.0, T_MAX, NUM_BUCKETS + 1)
else:
    raise ValueError('BUCKET_STYLE must be "log" or "uniform"')
assert len(BUCKET_EDGES) - 1 == NUM_BUCKETS

bucket_info = pd.DataFrame({
    "bucket": np.arange(NUM_BUCKETS),
    "start_s": BUCKET_EDGES[:-1],
    "end_s": BUCKET_EDGES[1:],
})
bucket_info["width_s"] = bucket_info["end_s"] - bucket_info["start_s"]
print(f"Bucket style: {BUCKET_STYLE} | window: first {T_MAX}s | {NUM_BUCKETS} buckets")
if BUCKET_STYLE == "log":
    print(f"ratio between consecutive edges: {BUCKET_EDGES[2] / BUCKET_EDGES[1]:.3f}")
display(bucket_info.round(3))

# --- FEATURE FUNCTIONS ------------------------------------------------------
def _count(idx, mask):
    return np.bincount(idx[mask], minlength=NUM_BUCKETS).astype(float)

def _sum(idx, mask, vals):
    return np.bincount(idx[mask], weights=vals[mask], minlength=NUM_BUCKETS)

def _mean_std(idx, mask, vals):
    """per-bucket mean and (population) std, 0 where the bucket is empty"""
    n  = _count(idx, mask)
    s  = _sum(idx, mask, vals)
    s2 = _sum(idx, mask, vals ** 2)
    mean = np.divide(s, n, out=np.zeros(NUM_BUCKETS), where=n > 0)
    var = np.divide(s2, n, out=np.zeros(NUM_BUCKETS), where=n > 0) - mean ** 2
    return mean, np.sqrt(np.maximum(var, 0))

N_FEATS = 8 * NUM_BUCKETS

def bucket_features_log(packet_seq):
    """
    one flow -> concatenated array in this order (each block has NUM_BUCKETS values):
      bytes_out, bytes_in, packets_out, packets_in,
      gap_mean, gap_std, size_mean, size_std
    """
    if len(packet_seq) == 0:
        return np.zeros(N_FEATS)

    arr = np.asarray(packet_seq, dtype=float)             # columns: timestamp, size, direction
    arr = arr[np.argsort(arr[:, 0], kind="stable")]       # make sure packets are time-ordered
    t, size, direction = arr[:, 0] - arr[0, 0], arr[:, 1], arr[:, 2]

    # bucket index of every packet; packets at/after T_MAX get index == NUM_BUCKETS and are dropped
    idx = np.searchsorted(BUCKET_EDGES, t, side="right") - 1
    keep = idx < NUM_BUCKETS
    is_out = keep & (direction > 0)
    is_in  = keep & (direction <= 0)

    bytes_out, bytes_in = _sum(idx, is_out, size), _sum(idx, is_in, size)
    pkts_out,  pkts_in  = _count(idx, is_out),     _count(idx, is_in)

    size_mean, size_std = _mean_std(idx, keep, size)

    # inter-arrival gap belongs to the bucket of the later packet (same as the old code)
    if len(t) >= 2:
        gaps = np.diff(t)
        gap_mean, gap_std = _mean_std(idx[1:], keep[1:], gaps)
    else:
        gap_mean = gap_std = np.zeros(NUM_BUCKETS)

    return np.concatenate([bytes_out, bytes_in, pkts_out, pkts_in,
                           gap_mean, gap_std, size_mean, size_std])


# --- BUILD THE FEATURE TABLE ------------------------------------------------
# keep the old uniform-bucket table around for comparison, if it exists
if "df_clean_features" in globals():
    df_clean_features_uniform = df_clean_features

feat = np.vstack([bucket_features_log(s) for s in df_clean["packet_seq"]])
B = NUM_BUCKETS
_names = lambda p: [f"{p}_{i}" for i in range(B)]

bytes_cols   = _names("bytes_out")   + _names("bytes_in")
packets_cols = _names("packets_out") + _names("packets_in")
timing_cols  = _names("gap_mean")    + _names("gap_std")
size_cols    = _names("size_mean")   + _names("size_std")

df_bytes   = pd.DataFrame(feat[:, 0 * B:2 * B], columns=bytes_cols,   index=df_clean.index)
df_packets = pd.DataFrame(feat[:, 2 * B:4 * B], columns=packets_cols, index=df_clean.index)
df_timing  = pd.DataFrame(feat[:, 4 * B:6 * B], columns=timing_cols,  index=df_clean.index)
df_size    = pd.DataFrame(feat[:, 6 * B:8 * B], columns=size_cols,    index=df_clean.index)

print(f"before shape: {df_clean.shape}")
df_clean_features = pd.concat([df_clean, df_bytes, df_packets, df_timing, df_size], axis=1)
print(f"after shape: {df_clean_features.shape}")

Bucket style: log | window: first 2s | 10 buckets
ratio between consecutive edges: 1.802


,bucket,start_s,end_s,width_s
0,0,0.000,0.010,0.010
1,1,0.010,0.018,0.008
2,2,0.018,0.032,0.014
3,3,0.032,0.058,0.026
4,4,0.058,0.105,0.047
5,5,0.105,0.190,0.084
6,6,0.190,0.342,0.152
7,7,0.342,0.616,0.274
8,8,0.616,1.110,0.494
9,9,1.110,2.000,0.890


before shape: (188670, 10)
after shape: (188670, 90)


In [87]:
# cell 3
# zs's code below
#ADJUST THE BUCKETS
#300s coverage: 30 * 10
BUCKET_SIZE_SECONDS = 30
NUM_BUCKETS = 10 

#bytes sent and received per bucket
def bucket_bytes_by_direction(packet_seq, bucket_size=BUCKET_SIZE_SECONDS, num_buckets=NUM_BUCKETS):
    if len(packet_seq) == 0:
        return np.zeros(num_buckets * 2)

    flow_start = packet_seq[0][0]
    bytes_out = np.zeros(num_buckets)
    bytes_in = np.zeros(num_buckets)

    for timestamp, size, direction in packet_seq:
        elapsed = timestamp - flow_start
        bucket_index = int(elapsed // bucket_size)
        if bucket_index < num_buckets:
            if direction > 0:
                bytes_out[bucket_index] += size
            else:
                bytes_in[bucket_index] += size

    return np.concatenate([bytes_out, bytes_in])

#packets sent and received per bucket
def bucket_packets_by_direction(packet_seq, bucket_size=BUCKET_SIZE_SECONDS, num_buckets=NUM_BUCKETS):
    if len(packet_seq) == 0:
        return np.zeros(num_buckets * 2)

    flow_start = packet_seq[0][0]
    packets_out = np.zeros(num_buckets)
    packets_in = np.zeros(num_buckets)

    for timestamp, size, direction in packet_seq:
        elapsed = timestamp - flow_start
        bucket_index = int(elapsed // bucket_size)
        if bucket_index < num_buckets:
            if direction > 0:
                packets_out[bucket_index] += 1
            else:
                packets_in[bucket_index] += 1

    return np.concatenate([packets_out, packets_in])

#mean and std of time between consecutive packets per bucket
def bucket_timing_stats(packet_seq, bucket_size=BUCKET_SIZE_SECONDS, num_buckets=NUM_BUCKETS):
    if len(packet_seq) < 2:
        return np.zeros(num_buckets * 2)  #mean and std per bucket

    flow_start = packet_seq[0][0]
    bucket_gaps = [[] for _ in range(num_buckets)]

    timestamps = [p[0] for p in packet_seq]
    for i in range(1, len(timestamps)):
        gap = timestamps[i] - timestamps[i-1]
        elapsed = timestamps[i] - flow_start
        bucket_index = int(elapsed // bucket_size)
        if bucket_index < num_buckets:
            bucket_gaps[bucket_index].append(gap)

    mean_gaps = np.array([np.mean(g) if g else 0 for g in bucket_gaps])
    std_gaps = np.array([np.std(g) if g else 0 for g in bucket_gaps])

    return np.concatenate([mean_gaps, std_gaps])

#mean and std of bytes of each packet per bucket
def bucket_size_stats(packet_seq, bucket_size=BUCKET_SIZE_SECONDS, num_buckets=NUM_BUCKETS):
    if len(packet_seq) == 0:
        return np.zeros(num_buckets * 2)  #mean and std per bucket

    flow_start = packet_seq[0][0]
    bucket_sizes = [[] for _ in range(num_buckets)]

    for timestamp, size, direction in packet_seq:
        elapsed = timestamp - flow_start
        bucket_index = int(elapsed // bucket_size)
        if bucket_index < num_buckets:
            bucket_sizes[bucket_index].append(size)

    mean_sizes = np.array([np.mean(s) if s else 0 for s in bucket_sizes])
    std_sizes = np.array([np.std(s) if s else 0 for s in bucket_sizes])

    return np.concatenate([mean_sizes, std_sizes])


In [88]:
feature_cols = bytes_cols + packets_cols + timing_cols + size_cols
X = df_clean_features[feature_cols]
y = df_clean_features["network_type"]

print(X.shape)            # should be (number of flows, 80)
print(y.value_counts())   # see how small Tor really is

(188670, 80)
network_type
non_vpn    169585
vpn         17769
tor          1316
Name: count, dtype: int64


In [89]:
# testing group shuffle split
# from sklearn.model_selection import GroupShuffleSplit

# #use stream_id for grouping the split
# groups = df_clean_features["stream_id"]

# #split 1: 70% train, 30% temp, grouped by stream_id
# gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=4266)
# train_idx, temp_idx = next(gss.split(X, y, groups=groups))

# X_train, X_temp = X.iloc[train_idx], X.iloc[temp_idx]
# y_train, y_temp = y.iloc[train_idx], y.iloc[temp_idx]
# groups_temp = groups.iloc[temp_idx]

# #split 2: temp into 50% validation, 50% test, grouped by stream_id
# gss_temp = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=4266)
# val_idx, test_idx = next(gss_temp.split(X_temp, y_temp, groups=groups_temp))

# X_val, X_test = X_temp.iloc[val_idx], X_temp.iloc[test_idx]
# y_val, y_test = y_temp.iloc[val_idx], y_temp.iloc[test_idx]

# #for verification
# groups_train = groups.iloc[train_idx]
# groups_val = groups_temp.iloc[val_idx]
# groups_test = groups_temp.iloc[test_idx]


In [90]:
# # NOW WITH THE DATA AUGMENTATION FUNCTIONS
# rng = np.random.default_rng(4266)
# WINDOW = BUCKET_SIZE_SECONDS * NUM_BUCKETS   # 300s, same window as the features

# def aug_window_shift(seq, window=WINDOW):
#     """Start the window at a random later point in a long flow (real, unused data)."""
#     t0, t_end = seq[0][0], seq[-1][0]
#     if t_end - t0 <= window:
#         return None
#     start = rng.uniform(t0, t_end - window)
#     sub = [p for p in seq if p[0] >= start]
#     return sub if len(sub) >= 2 else None

# def aug_time_jitter(seq, scale_range=(0.9, 1.1), noise=0.05):
#     """Mildly stretch/shrink the gaps between packets."""
#     times = np.array([p[0] for p in seq])
#     gaps = np.diff(times, prepend=times[0])
#     factor = rng.uniform(*scale_range)
#     gaps = gaps * factor * rng.normal(1, noise, len(gaps)).clip(0.5, 1.5)
#     new_times = times[0] + np.cumsum(gaps)
#     return [(t, s, d) for t, (_, s, d) in zip(new_times, seq)]

# def aug_size_jitter(seq, max_delta=8, min_size=54, max_size=1514):
#     """Nudge packet sizes slightly. NOT used for Tor (fixed 512-byte cells)."""
#     return [(t, int(np.clip(s + rng.integers(-max_delta, max_delta + 1), min_size, max_size)), d)
#             for t, s, d in seq]

# def aug_packet_drop(seq, p=0.03):
#     """Randomly drop ~3% of packets (simulated loss)."""
#     if len(seq) < 5:
#         return list(seq)
#     keep = rng.random(len(seq)) > p
#     keep[0] = True
#     return [pkt for pkt, k in zip(seq, keep) if k]

# def augment_flow(seq, is_tor=False):
#     s = list(seq)
#     if rng.random() < 0.5:
#         shifted = aug_window_shift(s)
#         if shifted:
#             s = shifted
#     s = aug_time_jitter(s)
#     if not is_tor:                    # protect Tor's fixed cell sizes
#         s = aug_size_jitter(s)
#     s = aug_packet_drop(s)
#     return s

# def flow_features(seq):
#     """Recompute the 80 features, same order as X."""
#     return np.concatenate([
#         bucket_bytes_by_direction(seq),
#         bucket_packets_by_direction(seq),
#         bucket_timing_stats(seq),
#         bucket_size_stats(seq),
#     ])

In [91]:
# # AUGMENTING TRAINING SET ONLY
# TARGET_RATIO = 0.5    # grow small classes to 50% of the biggest class...
# MAX_MULTIPLIER = 5    # ...but never more than 5x their original size
# MIN_PACKETS = 5       # only augment flows with enough packets

# train_df = df_clean_features.iloc[train_idx]
# counts = train_df["network_type"].value_counts()
# print("Before:\n", counts, "\n")
# print("Tor streams in train:",
#       train_df[train_df["network_type"].str.lower().str.contains("tor")]["stream_id"].nunique())

# aug_rows, aug_labels = [], []
# for cls, n in counts.items():
#     target = min(int(counts.max() * TARGET_RATIO), n * MAX_MULTIPLIER)
#     n_needed = target - n
#     if n_needed <= 0:
#         continue
#     pool = train_df[(train_df["network_type"] == cls) & (train_df["packet_count"] >= MIN_PACKETS)]
#     if len(pool) == 0:
#         continue
#     is_tor = "tor" in str(cls).lower()
#     sampled = pool.sample(n_needed, replace=True, random_state=4266)
#     for seq in sampled["packet_seq"]:
#         aug_rows.append(flow_features(augment_flow(seq, is_tor=is_tor)))
#         aug_labels.append(cls)

# X_aug = pd.DataFrame(aug_rows, columns=feature_cols)
# y_aug = pd.Series(aug_labels, name="network_type")

# X_train_aug = pd.concat([X_train.reset_index(drop=True), X_aug], ignore_index=True)
# y_train_aug = pd.concat([y_train.reset_index(drop=True), y_aug], ignore_index=True)

# print("\nAfter:\n", y_train_aug.value_counts())

In [92]:
# # FITTING ON TRAIN DATA ONLY
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(np.log1p(X_train_aug))
# X_val_scaled   = scaler.transform(np.log1p(X_val))
# X_test_scaled  = scaler.transform(np.log1p(X_test))


In [93]:
# ## cell 9 butb with this extra first line
# AUGMENT_CLASSES = {"tor"}
# TARGET_RATIO = 0.5    # grow small classes to 50% of the biggest class...
# MAX_MULTIPLIER = 5    # ...but never more than 5x their original size
# MIN_PACKETS = 5       # only augment flows with enough packets

# train_df = df_clean_features.iloc[train_idx]
# counts = train_df["network_type"].value_counts()
# print("Before:\n", counts, "\n")
# print("Tor streams in train:",
#       train_df[train_df["network_type"].str.lower().str.contains("tor")]["stream_id"].nunique())

# aug_rows, aug_labels = [], []
# for cls, n in counts.items():
#     target = min(int(counts.max() * TARGET_RATIO), n * MAX_MULTIPLIER)
#     n_needed = target - n
#     if n_needed <= 0:
#         continue
#     pool = train_df[(train_df["network_type"] == cls) & (train_df["packet_count"] >= MIN_PACKETS)]
#     if len(pool) == 0:
#         continue
#     is_tor = "tor" in str(cls).lower()
#     sampled = pool.sample(n_needed, replace=True, random_state=4266)
#     for seq in sampled["packet_seq"]:
#         aug_rows.append(flow_features(augment_flow(seq, is_tor=is_tor)))
#         aug_labels.append(cls)

# X_aug = pd.DataFrame(aug_rows, columns=feature_cols)
# y_aug = pd.Series(aug_labels, name="network_type")

# X_train_aug = pd.concat([X_train.reset_index(drop=True), X_aug], ignore_index=True)
# y_train_aug = pd.concat([y_train.reset_index(drop=True), y_aug], ignore_index=True)

# print("\nAfter:\n", y_train_aug.value_counts())

In [94]:
# fopr myself to check if the secong plan works
for cls in ["tor", "vpn", "non_vpn"]:
    sizes = (df_clean_features[df_clean_features["network_type"] == cls]
             .groupby("stream_id").size().sort_values(ascending=False))
    print(f"\n=== {cls}: {len(sizes)} streams ===")
    print("Biggest stream's share of class:", round(sizes.iloc[0] / sizes.sum() * 100, 1), "%")
    print(sizes.head(5))


=== tor: 51 streams ===
Biggest stream's share of class: 79.4 %
stream_id
Torrent01      1045
torYoutube2      56
torYoutube1      41
torYoutube3      21
torVimeo1        10
dtype: int64

=== vpn: 23 streams ===
Biggest stream's share of class: 27.8 %
stream_id
vpn_hangouts_audio2    4948
vpn_hangouts_audio1    3172
vpn_hangouts_chat1     2751
vpn_facebook_audio2    1335
vpn_facebook_chat1     1159
dtype: int64

=== non_vpn: 87 streams ===
Biggest stream's share of class: 25.6 %
stream_id
hangouts_audio4    43481
facebook_audio4    42805
skype_file8        30555
skype_audio3       20559
scp1                6846
dtype: int64


In [95]:
# trying a third way: capping flows per stream
stream_info = df_clean_features.groupby(["network_type", "stream_id"]).size()
for cap in [100, 200, 500, 1000]:
    print(cap, stream_info.clip(upper=cap).groupby(level=0).sum().to_dict())

## i cap flows per stream here
MAX_FLOWS_PER_STREAM = 200

# preview class sizes under different caps (useful for the group)
stream_info = df_clean_features.groupby(["network_type", "stream_id"]).size()
print("Flows per class under different caps:")
for cap in [100, 200, 500, 1000, None]:
    counts = stream_info if cap is None else stream_info.clip(upper=cap)
    print(f"  cap={cap}:", counts.groupby(level=0).sum().to_dict())

# shuffle, then keep at most N flows per stream (= random sample per stream)
df_capped = (df_clean_features
             .sample(frac=1, random_state=4266)
             .groupby("stream_id")
             .head(MAX_FLOWS_PER_STREAM)
             .reset_index(drop=True))

print(f"\nUsing cap = {MAX_FLOWS_PER_STREAM}")
print(df_capped["network_type"].value_counts())
tor_streams = df_capped[df_capped["network_type"] == "tor"]["stream_id"].value_counts()
print("Biggest Tor stream's share now:", round(tor_streams.iloc[0] / tor_streams.sum() * 100, 1), "%")


# features + stratified group split here first
from sklearn.model_selection import StratifiedGroupKFold

feature_cols = bytes_cols + packets_cols + timing_cols + size_cols
X = df_capped[feature_cols]
y = df_capped["network_type"]
groups = df_capped["stream_id"]

def make_split(seed):
    sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)
    tv, te = next(sgkf.split(X, y, groups))
    sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)
    tr, va = next(sgkf2.split(X.iloc[tv], y.iloc[tv], groups.iloc[tv]))
    return tv[tr], tv[va], te

def split_error(tr, va, te):
    """Worst-case gap between any class's split % and the 72/14/14 target."""
    pct = pd.DataFrame({"train": y.iloc[tr].value_counts(),
                        "val": y.iloc[va].value_counts(),
                        "test": y.iloc[te].value_counts()}).fillna(0)
    pct = pct.div(pct.sum(axis=1), axis=0)
    target = pd.Series({"train": 0.72, "val": 0.14, "test": 0.14})
    return (pct - target).abs().max().max()

errors = {s: split_error(*make_split(s)) for s in range(100)}
best_seed = min(errors, key=errors.get)
print(f"Best seed: {best_seed} (worst-case gap: {errors[best_seed]:.1%})")

train_idx, val_idx, test_idx = make_split(best_seed)
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val,   y_val   = X.iloc[val_idx],   y.iloc[val_idx]
X_test,  y_test  = X.iloc[test_idx],  y.iloc[test_idx]

# check: counts and % per split
dist = pd.DataFrame({"train": y_train.value_counts(),
                     "val": y_val.value_counts(),
                     "test": y_test.value_counts()}).fillna(0).astype(int)
print(dist, "\n")
print((dist.div(dist.sum(axis=1), axis=0) * 100).round(1), "\n")

# check: no stream in two splits (should be 0 0 0)
g_tr, g_va, g_te = set(groups.iloc[train_idx]), set(groups.iloc[val_idx]), set(groups.iloc[test_idx])
print("Overlaps:", len(g_tr & g_va), len(g_tr & g_te), len(g_va & g_te))

100 {'non_vpn': 6112, 'tor': 371, 'vpn': 1808}
200 {'non_vpn': 9687, 'tor': 471, 'vpn': 3118}
500 {'non_vpn': 15300, 'tor': 771, 'vpn': 6118}
1000 {'non_vpn': 20790, 'tor': 1271, 'vpn': 9404}
Flows per class under different caps:
  cap=100: {'non_vpn': 6112, 'tor': 371, 'vpn': 1808}
  cap=200: {'non_vpn': 9687, 'tor': 471, 'vpn': 3118}
  cap=500: {'non_vpn': 15300, 'tor': 771, 'vpn': 6118}
  cap=1000: {'non_vpn': 20790, 'tor': 1271, 'vpn': 9404}
  cap=None: {'non_vpn': 169585, 'tor': 1316, 'vpn': 17769}

Using cap = 200
network_type
non_vpn    9687
vpn        3118
tor         471
Name: count, dtype: int64
Biggest Tor stream's share now: 42.5 %
Best seed: 2 (worst-case gap: 3.4%)
              train   val  test
network_type                   
non_vpn        6939  1196  1552
vpn            2235   342   541
tor             331    61    79 

              train   val  test
network_type                   
non_vpn        71.6  12.3  16.0
vpn            71.7  11.0  17.4
tor            70.3  1

In [96]:
## augmentation function for the cell above
## this is cell 8
rng = np.random.default_rng(4266)
WINDOW = T_MAX   # features now only see the first T_MAX seconds

def aug_window_shift(seq, window=WINDOW):
    t0, t_end = seq[0][0], seq[-1][0]
    if t_end - t0 <= window:
        return None
    start = rng.uniform(t0, t_end - window)
    sub = [p for p in seq if p[0] >= start]
    return sub if len(sub) >= 2 else None

def aug_time_jitter(seq, scale_range=(0.9, 1.1), noise=0.05):
    times = np.array([p[0] for p in seq])
    gaps = np.diff(times, prepend=times[0])
    factor = rng.uniform(*scale_range)
    gaps = gaps * factor * rng.normal(1, noise, len(gaps)).clip(0.5, 1.5)
    new_times = times[0] + np.cumsum(gaps)
    return [(t, s, d) for t, (_, s, d) in zip(new_times, seq)]

def aug_size_jitter(seq, max_delta=8, min_size=54, max_size=1514):
    return [(t, int(np.clip(s + rng.integers(-max_delta, max_delta + 1), min_size, max_size)), d)
            for t, s, d in seq]

def aug_packet_drop(seq, p=0.03):
    if len(seq) < 5:
        return list(seq)
    keep = rng.random(len(seq)) > p
    keep[0] = True
    return [pkt for pkt, k in zip(seq, keep) if k]

def augment_flow(seq, is_tor=False):
    s = list(seq)
    if rng.random() < 0.5:
        shifted = aug_window_shift(s)
        if shifted:
            s = shifted
    s = aug_time_jitter(s)
    if not is_tor:            # protect Tor's fixed 512-byte cell sizes
        s = aug_size_jitter(s)
    s = aug_packet_drop(s)
    return s

def flow_features(seq):
    return bucket_features_log(seq)

In [97]:
# augment tor in the training set only
# this is cell 9

AUGMENT_CLASSES = {"tor"}   # VPN handled by class weights
TARGET_RATIO = 0.5          # grow to at most 50% of the biggest class...
MAX_MULTIPLIER = 5          # ...and at most 5x original size
MIN_PACKETS = 5

train_df = df_capped.iloc[train_idx]
counts = train_df["network_type"].value_counts()
print("Before:\n", counts, "\n")

aug_rows, aug_labels = [], []
for cls in AUGMENT_CLASSES:
    if cls not in counts:
        continue
    n = counts[cls]
    n_needed = min(int(counts.max() * TARGET_RATIO), n * MAX_MULTIPLIER) - n
    pool = train_df[(train_df["network_type"] == cls) & (train_df["packet_count"] >= MIN_PACKETS)]
    if n_needed <= 0 or len(pool) == 0:
        continue

    print(f"{cls}: {pool['stream_id'].nunique()} streams in pool, "
          f"{(pool['flow_duration'] > WINDOW).mean():.1%} of flows longer than {WINDOW}s")

    # every stream gets an equal chance, so one big stream can't dominate
    stream_sizes = pool["stream_id"].map(pool["stream_id"].value_counts())
    sampled = pool.sample(n_needed, replace=True, weights=1 / stream_sizes, random_state=4266)

    is_tor = "tor" in str(cls).lower()
    for seq in sampled["packet_seq"]:
        aug_rows.append(flow_features(augment_flow(seq, is_tor=is_tor)))
        aug_labels.append(cls)

X_aug = pd.DataFrame(aug_rows, columns=feature_cols)
y_aug = pd.Series(aug_labels, name="network_type")

X_train_aug = pd.concat([X_train.reset_index(drop=True), X_aug], ignore_index=True)
y_train_aug = pd.concat([y_train.reset_index(drop=True), y_aug], ignore_index=True)
print("\nAfter:\n", y_train_aug.value_counts())

Before:
 network_type
non_vpn    6939
vpn        2235
tor         331
Name: count, dtype: int64 

tor: 36 streams in pool, 70.7% of flows longer than 2s

After:
 network_type
non_vpn    6939
vpn        2235
tor        1655
Name: count, dtype: int64


In [98]:
## scale to fit on train only. 
# cell number ten
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(np.log1p(X_train_aug))
X_val_scaled   = scaler.transform(np.log1p(X_val))
X_test_scaled  = scaler.transform(np.log1p(X_test))
print("Train:", X_train_scaled.shape, "| Val:", X_val_scaled.shape, "| Test:", X_test_scaled.shape)

Train: (10829, 80) | Val: (1599, 80) | Test: (2172, 80)


In [99]:
# A/b comparison to ttest if it works
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

def run(Xtr, ytr, name):
    sc = StandardScaler().fit(np.log1p(Xtr))                      # fit on this train set only
    model = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                   random_state=4266, n_jobs=-1)
    model.fit(sc.transform(np.log1p(Xtr)), ytr)
    pred = model.predict(sc.transform(np.log1p(X_val)))
    print(f"\n=== {name} | macro-F1: {f1_score(y_val, pred, average='macro'):.3f} ===")
    print(classification_report(y_val, pred, digits=3))

run(X_train, y_train, "A: no augmentation (class weights only)")
run(X_train_aug, y_train_aug, "B: Tor augmentation + class weights")


=== A: no augmentation (class weights only) | macro-F1: 0.677 ===
              precision    recall  f1-score   support

     non_vpn      0.947     0.900     0.923      1196
         tor      0.144     0.262     0.186        61
         vpn      0.909     0.936     0.922       342

    accuracy                          0.883      1599
   macro avg      0.667     0.699     0.677      1599
weighted avg      0.908     0.883     0.895      1599


=== B: Tor augmentation + class weights | macro-F1: 0.695 ===
              precision    recall  f1-score   support

     non_vpn      0.954     0.895     0.923      1196
         tor      0.176     0.361     0.237        61
         vpn      0.912     0.939     0.925       342

    accuracy                          0.884      1599
   macro avg      0.681     0.731     0.695      1599
weighted avg      0.915     0.884     0.897      1599



In [100]:
# find out why it somehow works but at the expense of VPN data
from sklearn.metrics import confusion_matrix, recall_score

def fit_predict(Xtr, ytr, seed=4266):
    sc = StandardScaler().fit(np.log1p(Xtr))
    m = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=seed, n_jobs=-1)
    m.fit(sc.transform(np.log1p(Xtr)), ytr)
    return m.predict(sc.transform(np.log1p(X_val)))

labels = ["non_vpn", "vpn", "tor"]
versions = {"A (no aug)": (X_train, y_train), "B (aug)": (X_train_aug, y_train_aug)}

# 1. CONFUSION MATRIX: where do Tor flows go, and where did VPN recall go?
for name, (Xtr, ytr) in versions.items():
    cm = confusion_matrix(y_val, fit_predict(Xtr, ytr), labels=labels)
    print(f"\n=== Confusion matrix: {name} (rows = true, cols = predicted) ===")
    print(pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels]))

# 2. WHICH TOR APPS are in train vs val? (tests the 'Torrent dominates' theory)
print("\n=== Tor streams in TRAIN (top 10) ===")
print(df_capped.iloc[train_idx].query("network_type == 'tor'")["stream_id"].value_counts().head(10))

val_df = df_capped.iloc[val_idx].copy()
val_df["pred"] = fit_predict(X_train_aug, y_train_aug)
tor_val = val_df[val_df["network_type"] == "tor"].assign(caught=lambda d: d["pred"].eq("tor"))
print("\n=== Tor streams in VAL: how many flows the model (B) caught ===")
print(tor_val.groupby("stream_id").agg(flows=("caught", "size"), caught=("caught", "sum"))
             .sort_values("flows", ascending=False))

# 3. IS THE TOR GAIN REAL OR LUCK? Repeat with 5 model seeds
rows = []
for seed in range(5):
    for name, (Xtr, ytr) in versions.items():
        pred = fit_predict(Xtr, ytr, seed)
        rows.append({"model": name, "macro_f1": f1_score(y_val, pred, average="macro"),
                     "tor_recall": recall_score(y_val, pred, labels=["tor"], average="macro"),
                     "vpn_recall": recall_score(y_val, pred, labels=["vpn"], average="macro")})
print("\n=== Average over 5 seeds (mean, std) ===")
print(pd.DataFrame(rows).groupby("model").agg(["mean", "std"]).round(3))




=== Confusion matrix: A (no aug) (rows = true, cols = predicted) ===
              pred_non_vpn  pred_vpn  pred_tor
true_non_vpn          1076        27        93
true_vpn                20       320         2
true_tor                40         5        16

=== Confusion matrix: B (aug) (rows = true, cols = predicted) ===
              pred_non_vpn  pred_vpn  pred_tor
true_non_vpn          1070        27        99
true_vpn                17       321         4
true_tor                35         4        22

=== Tor streams in TRAIN (top 10) ===
stream_id
Torrent01                    200
torYoutube3                   21
torVimeo1                     10
tor_spotify2-2                 7
AUDIO_tor_spotify2             7
torVimeo3                      6
torTwitter                     6
BROWSING_tor_browsing_mam      6
torGoogle                      5
BROWSING_tor_browsing_ger      5
Name: count, dtype: int64

=== Tor streams in VAL: how many flows the model (B) caught ===
                 

Undersample + LOGREG

In [101]:
# =====================================================================
# UNDERSAMPLE NON-VPN (train only) + LOGISTIC REGRESSION BASELINE
# Run after: new bucketing -> cap 200 -> balanced split
# Val/test are never undersampled.
# =====================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

train_df = df_capped.iloc[train_idx]
n_vpn    = (train_df["network_type"] == "vpn").sum()
n_nonvpn = (train_df["network_type"] == "non_vpn").sum()

def undersample_nonvpn(df, target, seed=4266):
    """Keep `target` non-VPN flows; each recording gets an equal chance. Other classes untouched."""
    nv = df[df["network_type"] == "non_vpn"]
    if target >= len(nv):
        return df
    w = 1 / nv["stream_id"].map(nv["stream_id"].value_counts())
    keep = nv.sample(target, weights=w, random_state=seed)
    return pd.concat([keep, df[df["network_type"] != "non_vpn"]])

setups = {
    "original":           n_nonvpn,     # no undersampling
    "non_vpn = 2 x vpn":  2 * n_vpn,
    "non_vpn = vpn":      n_vpn,
}

rows, fitted = [], {}
for name, target in setups.items():
    tr = undersample_nonvpn(train_df, target)
    Xtr, ytr = tr[feature_cols], tr["network_type"]
    sc = StandardScaler().fit(np.log1p(Xtr))                     # fit on THIS train set only
    Xtr_s, Xva_s = sc.transform(np.log1p(Xtr)), sc.transform(np.log1p(X_val))
    for cw in [None, "balanced"]:                                # with vs without class weights
        m = LogisticRegression(class_weight=cw, max_iter=5000).fit(Xtr_s, ytr)
        pred = m.predict(Xva_s)
        fitted[(name, cw or "none")] = (m, pred)
        rows.append({"undersampling": name, "class_weight": cw or "none",
                     "train sizes": ytr.value_counts().to_dict(),
                     "macro_F1":      f1_score(y_val, pred, average="macro"),
                     "tor_recall":    recall_score(y_val, pred, labels=["tor"], average="macro"),
                     "vpn_recall":    recall_score(y_val, pred, labels=["vpn"], average="macro"),
                     "nonvpn_recall": recall_score(y_val, pred, labels=["non_vpn"], average="macro")})

results = pd.DataFrame(rows).round(3)
display(results)

# ---- detail for the best setup ----
best = results.loc[results["macro_F1"].idxmax()]
m, pred = fitted[(best["undersampling"], best["class_weight"])]
print(f"\nBest: {best['undersampling']} | class_weight={best['class_weight']} | macro-F1 {best['macro_F1']:.3f}\n")
print(classification_report(y_val, pred, digits=3))

labels = ["non_vpn", "vpn", "tor"]
print(pd.DataFrame(confusion_matrix(y_val, pred, labels=labels),
                   index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels]))

# ---- which features push towards each class? ----
coef = pd.DataFrame(m.coef_, index=m.classes_, columns=feature_cols)
for cls in labels:
    print(f"\nTop 5 features pushing towards {cls}:")
    print(coef.loc[cls].sort_values(ascending=False).head(5).round(3))

,undersampling,class_weight,train sizes,macro_F1,tor_recall,vpn_recall,nonvpn_recall
0,original,none,"{'non_vpn': 6939, 'vpn': 2235, 'tor': 331}",0.671,0.131,0.860,0.967
1,original,balanced,"{'non_vpn': 6939, 'vpn': 2235, 'tor': 331}",0.554,0.279,0.874,0.640
2,non_vpn = 2 x vpn,none,"{'non_vpn': 4470, 'vpn': 2235, 'tor': 331}",0.647,0.131,0.880,0.924
3,non_vpn = 2 x vpn,balanced,"{'non_vpn': 4470, 'vpn': 2235, 'tor': 331}",0.559,0.279,0.880,0.647
4,non_vpn = vpn,none,"{'non_vpn': 2235, 'vpn': 2235, 'tor': 331}",0.600,0.131,0.909,0.849
5,non_vpn = vpn,balanced,"{'non_vpn': 2235, 'vpn': 2235, 'tor': 331}",0.558,0.279,0.874,0.651



Best: original | class_weight=none | macro-F1 0.671

              precision    recall  f1-score   support

     non_vpn      0.929     0.967     0.948      1196
         tor      0.364     0.131     0.193        61
         vpn      0.886     0.860     0.872       342

    accuracy                          0.912      1599
   macro avg      0.726     0.653     0.671      1599
weighted avg      0.898     0.912     0.903      1599

              pred_non_vpn  pred_vpn  pred_tor
true_non_vpn          1157        32         7
true_vpn                41       294         7
true_tor                47         6         8

Top 5 features pushing towards non_vpn:
size_mean_2     1.969
packets_in_0    1.597
gap_mean_0      0.886
packets_in_1    0.742
bytes_out_0     0.741
Name: non_vpn, dtype: float64

Top 5 features pushing towards vpn:
packets_out_0    1.720
size_std_2       1.220
gap_mean_0       1.215
bytes_in_6       1.206
gap_mean_2       0.982
Name: vpn, dtype: float64

Top 5 features pu

In [102]:
from sklearn.ensemble import RandomForestClassifier

# 1. full undersampling comparison (did undersampling help?)
display(results[["undersampling", "class_weight", "macro_F1", "tor_recall", "vpn_recall", "nonvpn_recall"]])
print(f"Always-guess-non_vpn accuracy: {y_val.value_counts(normalize=True).max():.3f}")

# 2. is 'direction' a shortcut? direction of each flow's FIRST packet, by class
first_dir = df_capped["packet_seq"].apply(lambda s: "out" if s[0][2] > 0 else "in")
print("\nFirst-packet direction by class:")
print(pd.crosstab(df_capped["network_type"], first_dir, normalize="index").round(3))

# 3. how much data is in the 3s window?
n_in_window = df_capped["packet_seq"].apply(lambda s: sum(1 for p in s if p[0] - s[0][0] < T_MAX))
print("\nPackets inside the first T_MAX seconds (per flow):")
print(n_in_window.groupby(df_capped["network_type"]).describe()[["mean", "25%", "50%", "75%"]])
print("\nShare of flows with only 1 packet in the window:")
print((n_in_window == 1).groupby(df_capped["network_type"]).mean().round(3))

# 4. features or model? Random Forest on the SAME new features + SAME undersampling
tr = undersample_nonvpn(train_df, 2 * n_vpn)
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=4266, n_jobs=-1)
rf.fit(tr[feature_cols], tr["network_type"])
print("\nRandom Forest, new 3s log buckets, non_vpn = 2 x vpn:")
print(classification_report(y_val, rf.predict(X_val), digits=3))

,undersampling,class_weight,macro_F1,tor_recall,vpn_recall,nonvpn_recall
0,original,none,0.671,0.131,0.860,0.967
1,original,balanced,0.554,0.279,0.874,0.640
2,non_vpn = 2 x vpn,none,0.647,0.131,0.880,0.924
3,non_vpn = 2 x vpn,balanced,0.559,0.279,0.880,0.647
4,non_vpn = vpn,none,0.600,0.131,0.909,0.849
5,non_vpn = vpn,balanced,0.558,0.279,0.874,0.651


Always-guess-non_vpn accuracy: 0.748

First-packet direction by class:
packet_seq       in    out
network_type              
non_vpn       0.118  0.882
tor           0.170  0.830
vpn           0.012  0.988

Packets inside the first T_MAX seconds (per flow):
                   mean  25%  50%   75%
network_type                           
non_vpn       73.629091  2.0  2.0   4.0
tor           71.552017  2.0  2.0  14.0
vpn           26.482040  2.0  2.0   4.0

Share of flows with only 1 packet in the window:
network_type
non_vpn    0.069
tor        0.180
vpn        0.053
Name: packet_seq, dtype: float64

Random Forest, new 3s log buckets, non_vpn = 2 x vpn:
              precision    recall  f1-score   support

     non_vpn      0.947     0.881     0.913      1196
         tor      0.135     0.279     0.182        61
         vpn      0.889     0.936     0.912       342

    accuracy                          0.870      1599
   macro avg      0.657     0.699     0.669      1599
weighted avg  